# Phase 05B: ExpansionNet v2 → S1 Feature-KD Compatibility GPU Preflight

Notebook này **không huấn luyện full run**. Mục tiêu là quyết định khách quan xem nhánh
Feature KD mới có đủ điều kiện đi tiếp sang Phase 05C hay phải dừng theo fallback đã
đăng ký ở Phase 05A.

Nguồn đặc trưng được khóa:

- Teacher: đầu ra `input_linear(swin_transf(image))`, kích thước `[B,144,512]`;
- căn chỉnh không gian: `12×12 → 7×7` bằng adaptive average pooling;
- projection học được: `Linear(512,256) + LayerNorm`, **không dùng identity**;
- Student target: visual memory S1 `[B,49,256]`;
- loss: bình phương khoảng cách giữa các vector đã L2-normalize;
- Teacher frozen và stop-gradient.

Điều kiện chính: với `λ_feature=0.1`, tỷ số chuẩn gradient có trọng số
`||∇(λL_feature)|| / ||∇L_caption||` phải nằm trong `[0.05, 0.50]`.

## Kaggle inputs

1. Phase 05A re-entry output sau thay Teacher A.
2. Phase 04B ExpansionNet v2 → S1 Sequence-KD preflight output.
3. Phase 03B ExpansionNet v2 offline preflight output.
4. Phase 02A shared Student bundle.
5. ExpansionNet v2 offline model package.
6. MS COCO 2014 images.

GPU T4 hoặc mạnh hơn, Internet Off. Chạy tuần tự Cell 1 → Cell 8.
Held-out Test luôn `SEALED`.


## Cell 1 — Environment, authorization và immutable contracts

Xác minh GPU, marker Phase 05A/04B/03B/02A, checksum và leakage boundary.
Chưa mở ảnh, chưa nạp model.


In [1]:
import csv
import gc
import hashlib
import json
import math
import os
import pickle
import platform
import random
import sys
import time
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
SEED = 42
LAMBDA_FEATURE = 0.1
GRADIENT_RATIO_MIN = 0.05
GRADIENT_RATIO_MAX = 0.50
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_INPUT_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
MODEL_INPUT_ROOT = (
    INPUT_ROOT / "models"
    if (INPUT_ROOT / "models").is_dir()
    else INPUT_ROOT
)
DATASET_INPUT_ROOT = (
    INPUT_ROOT / "datasets"
    if (INPUT_ROOT / "datasets").is_dir()
    else INPUT_ROOT
)
OUTPUT_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//phase5_teacher_a_expansionnet_s1_feature_preflight",
))
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
RESULT_DIR = OUTPUT_ROOT / "results"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, RESULT_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temp, path)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_INPUT_ROOT.rglob(name))
    assert len(matches) == 1, f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root)
    return target

def verify_csv_manifest(root, reference, expected_rows=None):
    path = resolve_inside(root, reference["relative_path"])
    assert sha256_file(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    declared_rows = reference.get("rows", expected_rows)
    if declared_rows is not None:
        assert len(rows) == int(declared_rows), (
            path, len(rows), int(declared_rows)
        )
    for row in rows:
        target = resolve_inside(root, row["relative_path"])
        assert target.is_file()
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256_file(target) == row["sha256"]
    return rows

def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

if not torch.cuda.is_available():
    raise RuntimeError("Phase 05B yêu cầu GPU. Bật Kaggle GPU T4 rồi chạy lại.")
set_deterministic(SEED)
device = torch.device("cuda")

P5A_COMPLETE_PATH = exactly_one("PHASE5_KD_IMPROVEMENT_REENTRY_COMPLETE.json")
P5A_AUTH_PATH = exactly_one(
    "PHASE5_TEACHER_A_EXPANSIONNET_S1_FEATURE_COMPATIBILITY_PREFLIGHT_AUTHORIZED.json"
)
P4B_COMPLETE_PATH = exactly_one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_PREFLIGHT_COMPLETE.json")
P3B_READY_PATH = exactly_one("PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json")
P2_READY_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")

p5a = read_json(P5A_COMPLETE_PATH)
auth = read_json(P5A_AUTH_PATH)
p4b = read_json(P4B_COMPLETE_PATH)
p3b = read_json(P3B_READY_PATH)
p2 = read_json(P2_READY_PATH)
assert p5a["status"] == "complete"
assert auth["status"] == "authorized"
assert auth["anchor_teacher"] == "Teacher A — ExpansionNet v2 pretrained"
assert auth["anchor_student"] == "S1 — MobileNetV3-Large seed 42"
assert auth["training_authorized"] is False
assert p4b["status"] == "complete" and p4b["method"] == "sequence_kd"
assert p3b["status"] == "ready" and p3b["teacher"] == "ExpansionNet v2"
assert p3b["direct_logit_kd_allowed"] is False
assert p3b["feature_kd_requires_projection"] is True
assert p2["status"] == "ready"
for marker in [p5a, auth, p4b, p3b]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

P5A_ROOT, P4B_ROOT, P3B_ROOT, P2_ROOT = (
    P5A_COMPLETE_PATH.parent, P4B_COMPLETE_PATH.parent,
    P3B_READY_PATH.parent, P2_READY_PATH.parent,
)
p5a_rows = verify_csv_manifest(
    P5A_ROOT, p5a["manifest"]["csv"], p5a["manifest_rows"]
)
p4b_rows = verify_csv_manifest(P4B_ROOT, p4b["manifest"]["csv"])
p3b_rows = verify_csv_manifest(P3B_ROOT, p3b["manifest"]["csv"])
p2_rows = verify_csv_manifest(P2_ROOT, p2["manifests"]["csv"])

STUDY_PATH = resolve_inside(
    P5A_ROOT, "configs/phase5_kd_improvement_reentry_study_contract.json"
)
study = read_json(STUDY_PATH)
assert study["status"] == "provisional_locked"
candidate = study["feature_candidate"]
assert candidate["alignment"].startswith("explicit spatial alignment")
assert candidate["teacher_stop_gradient"] is True
assert float(candidate["lambda_feature_candidate"]) == LAMBDA_FEATURE
assert float(candidate["preflight_gradient_ratio_gate"]["minimum"]) == GRADIENT_RATIO_MIN
assert float(candidate["preflight_gradient_ratio_gate"]["maximum"]) == GRADIENT_RATIO_MAX

write_json(AUDIT_DIR / "phase5b_cell01_authorization.json", {
    "schema_version": "phase5b_cell01_v1", "created_at_utc": now_utc(),
    "status": "passed", "gpu": torch.cuda.get_device_name(0),
    "strict_deterministic": True, "phase5a_manifest_rows": len(p5a_rows),
    "phase4b_manifest_rows": len(p4b_rows), "phase3b_manifest_rows": len(p3b_rows),
    "phase2_manifest_rows": len(p2_rows), "lambda_feature": LAMBDA_FEATURE,
    "gradient_ratio_gate": [GRADIENT_RATIO_MIN, GRADIENT_RATIO_MAX],
    "dataset_read": False, "model_loaded": False, "training_started": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72)
print("PHASE 05B CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Phase 05A / 04B / 03B     : VERIFIED")
print("Manifests                 :", len(p5a_rows), len(p4b_rows), len(p3b_rows), len(p2_rows))
print("Lambda feature            :", LAMBDA_FEATURE)
print("Gradient-ratio gate       : [0.05, 0.50]")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 8 code cells")
print("Next                      : verify offline model and fixed batch")


PHASE 05B CELL 1 STATUS
Status                    : PASSED
GPU                       : Tesla T4
Phase 05A / 04B / 03B     : VERIFIED
Manifests                 : 13 8 14 38
Lambda feature            : 0.1
Gradient-ratio gate       : [0.05, 0.50]
Full training started     : False
Test Set status           : SEALED
Progress                  : 1 / 8 code cells
Next                      : verify offline model and fixed batch


## Cell 2 — Offline source, checkpoint và fixed development batch

Khóa checkpoint ExpansionNet v2, bộ 64 cặp supervision của Phase 04B và dựng
một batch 4 ảnh Train+Restval. Không đọc Test Parquet.


In [2]:
STARTED = time.perf_counter()
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

model_manifests = []
for path in MODEL_INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(path)
    except Exception:
        continue
    if payload.get("schema_version") == "expansionnet_v2_offline_bundle_v1":
        model_manifests.append((path.resolve(), payload))
assert len(model_manifests) == 1, model_manifests
MODEL_MANIFEST_PATH, model_manifest = model_manifests[0]
MODEL_ROOT = MODEL_MANIFEST_PATH.parent
assert model_manifest["upstream_commit"] == "eb7f1c98d00cdaa61b60c43852155b3742fe5b45"
assert model_manifest["doi"] == "10.1109/BigData59044.2023.10386812"

def verify_model_record(record):
    target = resolve_inside(MODEL_ROOT, record["relative_path"])
    assert target.is_file()
    assert target.stat().st_size == int(record["size_bytes"])
    assert sha256_file(target) == record["sha256"]
    return target

CHECKPOINT_PATH = verify_model_record(model_manifest["checkpoint"])
NATIVE_VOCAB_PATH = verify_model_record(model_manifest["vocabulary"])
for record in model_manifest["required_files"]:
    verify_model_record(record)
assert sha256_file(CHECKPOINT_PATH) == p3b["checkpoint_sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native = pickle.load(handle)
NATIVE_WORD2IDX = native["word2idx_dict"]
NATIVE_IDX2WORD = native["idx2word_list"]
assert len(NATIVE_WORD2IDX) == 10000

fixed_path = resolve_inside(P4B_ROOT, "data/phase4b_expansionnet_fixed64_pairs.json")
fixed_payload = read_json(fixed_path)
fixed_rows = fixed_payload["rows"]
assert len(fixed_rows) == 64
assert len({int(row["image_id"]) for row in fixed_rows}) == 64

def find_image_leaf(directory_name, prefix):
    found = []
    for path in DATASET_INPUT_ROOT.rglob(directory_name):
        if path.is_dir() and next(path.glob(prefix + "*.jpg"), None):
            found.append(path.resolve())
    found = sorted(set(found))
    assert len(found) == 1, found
    return found[0]

TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")
def resolve_image(row):
    name = Path(row["image_path"]).name
    leaf = TRAIN_LEAF if name.startswith("COCO_train2014_") else VAL_LEAF
    target = leaf / name
    assert target.is_file()
    return target

teacher_transform = transforms.Compose([
    transforms.Resize((384, 384), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
student_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

batch_rows = fixed_rows[:4]
teacher_images, student_images = [], []
for row in batch_rows:
    with Image.open(resolve_image(row)) as source:
        rgb = source.convert("RGB")
        teacher_images.append(teacher_transform(rgb))
        student_images.append(student_transform(rgb))
TEACHER_IMAGES = torch.stack(teacher_images).to(device)
STUDENT_IMAGES = torch.stack(student_images).to(device)

def pad_full(sequences):
    width = max(len(x) for x in sequences)
    result = torch.zeros(len(sequences), width, dtype=torch.long)
    for index, sequence in enumerate(sequences):
        result[index, :len(sequence)] = torch.tensor(sequence, dtype=torch.long)
    return result

HUMAN_FULL = pad_full([row["human_ids"] for row in batch_rows]).to(device)
HUMAN_INPUT = HUMAN_FULL[:, :-1]
HUMAN_LABELS = HUMAN_FULL[:, 1:]
HUMAN_MASK = HUMAN_INPUT.ne(0)

write_json(AUDIT_DIR / "phase5b_cell02_fixed_batch.json", {
    "schema_version": "phase5b_cell02_v1", "created_at_utc": now_utc(),
    "status": "passed", "checkpoint_sha256": sha256_file(CHECKPOINT_PATH),
    "source_commit": model_manifest["upstream_commit"], "fixed_corpus_rows": 64,
    "audited_batch_images": 4, "teacher_image_shape": list(TEACHER_IMAGES.shape),
    "student_image_shape": list(STUDENT_IMAGES.shape),
    "human_decoder_shape": list(HUMAN_INPUT.shape), "caption_text_read": False,
    "test_parquet_opened": False, "test_images_read": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72)
print("PHASE 05B CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("ExpansionNet checkpoint   : VERIFIED")
print("Fixed supervision         : 64 / 64 VERIFIED")
print("Audited batch             : 4 Train+Restval images")
print("Teacher images            :", list(TEACHER_IMAGES.shape))
print("Student images            :", list(STUDENT_IMAGES.shape))
print("Test Set status           : SEALED")
print("Progress                  : 2 / 8 code cells")
print("Next                      : reconstruct frozen teacher and S1")


PHASE 05B CELL 2 STATUS
Status                    : PASSED
ExpansionNet checkpoint   : VERIFIED
Fixed supervision         : 64 / 64 VERIFIED
Audited batch             : 4 Train+Restval images
Teacher images            : [4, 3, 384, 384]
Student images            : [4, 3, 224, 224]
Test Set status           : SEALED
Progress                  : 2 / 8 code cells
Next                      : reconstruct frozen teacher and S1


## Cell 3 — Reconstruct ExpansionNet v2 và S1

Nạp nghiêm ngặt checkpoint Teacher/Student. Teacher frozen; chưa tạo optimizer.


In [3]:
STARTED = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
assert SOURCE_ROOT.is_dir()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3, swin_embed_dim=192,
    swin_depths=[2, 2, 18, 2], swin_num_heads=[6, 12, 24, 48],
    swin_window_size=12, swin_mlp_ratio=4.0, swin_qkv_bias=True,
    swin_qk_scale=None, swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3, num_heads=8,
    ff=2048, num_exp_enc_list=[32, 64, 128, 256, 512], num_exp_dec=16,
    output_word2idx=NATIVE_WORD2IDX, output_idx2word=NATIVE_IDX2WORD,
    max_seq_len=74, drop_args=drop_args, rank=device,
)
teacher_payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
loaded = TEACHER.load_state_dict(teacher_payload["model_state_dict"], strict=True)
assert not loaded.missing_keys and not loaded.unexpected_keys
del teacher_payload
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)

decoder_contract = read_json(resolve_inside(
    P2_ROOT, "configs/decoder/shared_transformer_decoder_contract.json"
))["architecture"]

class SharedCaptionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(self.vocabulary_size, self.d_model, padding_idx=self.pad_id)
        self.position_embedding = nn.Embedding(self.maximum_decoder_steps, self.d_model)
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model, nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]), dropout=float(config["dropout"]),
            activation=config["activation"], batch_first=True, norm_first=True, bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(layer, num_layers=int(config["num_layers"]))
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocabulary_size))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(length, device=ids.device).unsqueeze(0).expand(ids.shape[0], -1)
        target = self.token_embedding(ids) * math.sqrt(self.d_model) + self.position_embedding(positions)
        target = self.embedding_dropout(target)
        causal = torch.triu(torch.ones(length, length, dtype=torch.bool, device=ids.device), diagonal=1)
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal, tgt_key_padding_mask=~mask.bool()
        )
        return F.linear(self.final_norm(hidden), self.token_embedding.weight, self.output_bias)

class S1VisualEncoderAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(self.projection(tokens) + self.visual_position_embedding)

STUDENT_ENCODER = S1VisualEncoderAdapter()
STUDENT_DECODER = SharedCaptionDecoder(decoder_contract)
enc_path = resolve_inside(P2_ROOT, p2["required_artifacts"]["s1_encoder_checkpoint"])
dec_path = resolve_inside(P2_ROOT, p2["required_artifacts"]["shared_decoder_checkpoint"])
with torch.serialization.safe_globals([TorchVersion]):
    enc_payload = torch.load(enc_path, map_location="cpu", weights_only=True)
    dec_payload = torch.load(dec_path, map_location="cpu", weights_only=True)
enc_load = STUDENT_ENCODER.load_state_dict(enc_payload["state_dict"], strict=True)
dec_load = STUDENT_DECODER.load_state_dict(dec_payload["state_dict"], strict=True)
assert not enc_load.missing_keys and not enc_load.unexpected_keys
assert not dec_load.missing_keys and not dec_load.unexpected_keys
del enc_payload, dec_payload
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)
assert sum(p.numel() for p in TEACHER.parameters()) == 233_803_076
assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 3_231_024
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5_759_328
assert sum(p.numel() for p in TEACHER.parameters() if p.requires_grad) == 0

write_json(AUDIT_DIR / "phase5b_cell03_models.json", {
    "schema_version": "phase5b_cell03_v1", "created_at_utc": now_utc(),
    "status": "passed", "teacher_parameters": 233803076,
    "student_parameters": 8990352, "teacher_frozen": True,
    "strict_checkpoint_loads": "3 / 3", "optimizer_created": False,
    "training_started": False, "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72)
print("PHASE 05B CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher parameters        : 233803076 | FROZEN")
print("Student parameters        : 8990352")
print("Strict checkpoint loads   : 3 / 3")
print("Optimizer created         : False")
print("Full training started     : False")
print("Progress                  : 3 / 8 code cells")
print("Next                      : audit feature source and alignment")


PHASE 05B CELL 3 STATUS
Status                    : PASSED
Teacher parameters        : 233803076 | FROZEN
Student parameters        : 8990352
Strict checkpoint loads   : 3 / 3
Optimizer created         : False
Full training started     : False
Progress                  : 3 / 8 code cells
Next                      : audit feature source and alignment


## Cell 4 — Feature source và explicit alignment

Kiểm tra hình dạng, stop-gradient, projection học được và khả năng lặp lại.


In [4]:
STARTED = time.perf_counter()
torch.cuda.reset_peak_memory_stats()

class ExpansionNetToS1Alignment(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(512, 256, bias=True)
        self.norm = nn.LayerNorm(256)

    def forward(self, teacher_tokens):
        assert teacher_tokens.shape[1:] == (144, 512)
        grid = teacher_tokens.transpose(1, 2).reshape(teacher_tokens.shape[0], 512, 12, 12)
        pooled = F.adaptive_avg_pool2d(grid, (7, 7))
        tokens = pooled.flatten(2).transpose(1, 2).contiguous()
        return self.norm(self.projection(tokens))

ALIGNMENT = ExpansionNetToS1Alignment().to(device)
assert ALIGNMENT.projection.in_features == 512
assert ALIGNMENT.projection.out_features == 256
assert sum(p.numel() for p in ALIGNMENT.parameters()) == 131_840

@torch.no_grad()
def teacher_source(images):
    swin_tokens = TEACHER.swin_transf(images)
    assert swin_tokens.shape[1:] == (144, 1536)
    tokens = TEACHER.input_linear(swin_tokens)
    assert tokens.shape[1:] == (144, 512)
    return tokens.detach()

TEACHER.eval()
STUDENT_ENCODER.eval()
ALIGNMENT.eval()
with torch.no_grad():
    source_1 = teacher_source(TEACHER_IMAGES)
    source_2 = teacher_source(TEACHER_IMAGES)
    aligned_1 = ALIGNMENT(source_1)
    aligned_2 = ALIGNMENT(source_2)
    student_1 = STUDENT_ENCODER(STUDENT_IMAGES)
    student_2 = STUDENT_ENCODER(STUDENT_IMAGES)
assert source_1.requires_grad is False
assert aligned_1.shape == student_1.shape == (4, 49, 256)
teacher_repeat = float((source_1 - source_2).abs().max().item())
alignment_repeat = float((aligned_1 - aligned_2).abs().max().item())
student_repeat = float((student_1 - student_2).abs().max().item())
assert teacher_repeat == alignment_repeat == student_repeat == 0.0
assert torch.isfinite(source_1).all() and torch.isfinite(aligned_1).all()

write_json(AUDIT_DIR / "phase5b_cell04_alignment.json", {
    "schema_version": "phase5b_cell04_v1", "created_at_utc": now_utc(),
    "status": "passed", "teacher_source": "input_linear(swin_transf(image))",
    "teacher_source_shape": list(source_1.shape),
    "spatial_alignment": "adaptive_avg_pool2d 12x12 -> 7x7",
    "projection": "Linear(512,256)+LayerNorm", "identity_projection": False,
    "alignment_parameters": 131840, "aligned_shape": list(aligned_1.shape),
    "student_target_shape": list(student_1.shape), "teacher_stop_gradient": True,
    "repeat_errors": {
        "teacher": teacher_repeat, "alignment": alignment_repeat, "student": student_repeat,
    },
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72)
print("PHASE 05B CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher source            : [B,144,512]")
print("Spatial alignment         : 12x12 -> 7x7")
print("Learned projection        : 512 -> 256 + LayerNorm")
print("Identity projection       : False")
print("Aligned / Student target  : [B,49,256] / [B,49,256]")
print("Teacher stop-gradient     : True")
print("Repeat maximum errors     : 0.0 / 0.0 / 0.0")
print("Progress                  : 4 / 8 code cells")
print("Next                      : verify normalized feature-loss semantics")


PHASE 05B CELL 4 STATUS
Status                    : PASSED
Teacher source            : [B,144,512]
Spatial alignment         : 12x12 -> 7x7
Learned projection        : 512 -> 256 + LayerNorm
Identity projection       : False
Aligned / Student target  : [B,49,256] / [B,49,256]
Teacher stop-gradient     : True
Repeat maximum errors     : 0.0 / 0.0 / 0.0
Progress                  : 4 / 8 code cells
Next                      : verify normalized feature-loss semantics


## Cell 5 — Normalized feature-loss semantics

Xác minh loss hữu hạn, bất biến theo độ lớn vector và bằng 0 khi hai biểu diễn trùng nhau.


In [5]:
STARTED = time.perf_counter()
def normalized_feature_loss(student_tokens, aligned_teacher_tokens, eps=1e-6):
    student_unit = F.normalize(student_tokens.float(), p=2, dim=-1, eps=eps)
    teacher_unit = F.normalize(aligned_teacher_tokens.detach().float(), p=2, dim=-1, eps=eps)
    return ((student_unit - teacher_unit) ** 2).sum(dim=-1).mean()

with torch.no_grad():
    feature_loss_value = normalized_feature_loss(student_1, aligned_1)
    scale_error = abs(
        float(normalized_feature_loss(student_1 * 7.0, aligned_1 * 0.25).item())
        - float(feature_loss_value.item())
    )
    identity_loss = float(normalized_feature_loss(student_1, student_1).item())
assert torch.isfinite(feature_loss_value)
assert 0.0 <= float(feature_loss_value.item()) <= 4.0
assert scale_error < 2e-6
assert identity_loss < 1e-10

write_json(AUDIT_DIR / "phase5b_cell05_feature_semantics.json", {
    "schema_version": "phase5b_cell05_v1", "created_at_utc": now_utc(),
    "status": "passed", "loss": "token-wise squared L2 distance after L2 normalization",
    "feature_loss": float(feature_loss_value.item()), "finite": True,
    "theoretical_range": [0.0, 4.0], "scale_invariance_error": scale_error,
    "identity_loss": identity_loss, "teacher_stop_gradient": True,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72)
print("PHASE 05B CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Normalized feature loss   :", float(feature_loss_value.item()))
print("Finite / range [0,4]      : True / PASSED")
print("Scale-invariance error    :", scale_error)
print("Identity loss             :", identity_loss)
print("Progress                  : 5 / 8 code cells")
print("Next                      : gradient-ratio compatibility gate")


PHASE 05B CELL 5 STATUS
Status                    : PASSED
Normalized feature loss   : 2.006793737411499
Finite / range [0,4]      : True / PASSED
Scale-invariance error    : 0.0
Identity loss             : 0.0
Progress                  : 5 / 8 code cells
Next                      : gradient-ratio compatibility gate


## Cell 6 — Prespecified gradient-ratio gate

Đo gradient trên cùng batch, cùng trạng thái Student. Không optimizer step.
Tỷ số chỉ tính trên tham số encoder S1 chung cho hai nhánh.


In [6]:
STARTED = time.perf_counter()
def caption_ce(logits, labels):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), labels.reshape(-1),
        ignore_index=0, reduction="mean",
    )

def grad_vector(loss, parameters):
    gradients = torch.autograd.grad(loss, parameters, retain_graph=True, allow_unused=True)
    parts = [g.detach().float().reshape(-1) for g in gradients if g is not None]
    assert parts
    return torch.cat(parts)

TEACHER.eval()
STUDENT_ENCODER.eval()
STUDENT_DECODER.eval()
ALIGNMENT.eval()
shared_parameters = [p for p in STUDENT_ENCODER.parameters() if p.requires_grad]

teacher_fixed = teacher_source(TEACHER_IMAGES)
aligned = ALIGNMENT(teacher_fixed)
student_memory = STUDENT_ENCODER(STUDENT_IMAGES)
caption_logits = STUDENT_DECODER(HUMAN_INPUT, student_memory, HUMAN_MASK)
base_loss = caption_ce(caption_logits, HUMAN_LABELS)
feature_loss = normalized_feature_loss(student_memory, aligned)
base_grad = grad_vector(base_loss, shared_parameters)
feature_grad = grad_vector(LAMBDA_FEATURE * feature_loss, shared_parameters)
base_norm = float(torch.linalg.vector_norm(base_grad).item())
weighted_feature_norm = float(torch.linalg.vector_norm(feature_grad).item())
gradient_ratio = weighted_feature_norm / max(base_norm, 1e-12)
finite_gradients = bool(
    torch.isfinite(base_grad).all().item() and torch.isfinite(feature_grad).all().item()
)
ratio_passed = bool(
    finite_gradients
    and base_norm > 0.0
    and weighted_feature_norm > 0.0
    and GRADIENT_RATIO_MIN <= gradient_ratio <= GRADIENT_RATIO_MAX
)
teacher_gradients = sum(1 for p in TEACHER.parameters() if p.grad is not None)
assert teacher_gradients == 0
assert finite_gradients

write_json(RESULT_DIR / "phase5b_gradient_ratio_result.json", {
    "schema_version": "phase5b_gradient_ratio_v1", "created_at_utc": now_utc(),
    "status": "passed" if ratio_passed else "failed",
    "lambda_feature": LAMBDA_FEATURE, "base_caption_ce": float(base_loss.item()),
    "normalized_feature_loss": float(feature_loss.item()),
    "base_encoder_gradient_norm": base_norm,
    "weighted_feature_encoder_gradient_norm": weighted_feature_norm,
    "weighted_feature_to_base_ratio": gradient_ratio,
    "accepted_interval": [GRADIENT_RATIO_MIN, GRADIENT_RATIO_MAX],
    "gate_passed": ratio_passed, "finite_gradients": finite_gradients,
    "teacher_gradient_tensors": teacher_gradients,
    "optimizer_step_performed": False, "training_started": False,
    "test_set_status": "sealed", "test_set_used": False,
})
write_json(AUDIT_DIR / "phase5b_cell06_gradient_ratio.json", {
    "schema_version": "phase5b_cell06_v1", "created_at_utc": now_utc(),
    "status": "passed" if ratio_passed else "failed",
    "gradient_ratio": gradient_ratio, "gate": [GRADIENT_RATIO_MIN, GRADIENT_RATIO_MAX],
    "gate_passed": ratio_passed, "optimizer_created": False,
    "optimizer_step_performed": False, "teacher_gradients": teacher_gradients,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72)
print("PHASE 05B CELL 6 GRADIENT-RATIO GATE")
print("=" * 72)
print("Base caption CE           :", float(base_loss.item()))
print("Normalized feature loss   :", float(feature_loss.item()))
print("Base encoder grad norm    :", base_norm)
print("Weighted feature grad norm:", weighted_feature_norm)
print("Gradient ratio            :", gradient_ratio)
print("Accepted interval         : [0.05, 0.50]")
print("Gate                      :", "PASSED" if ratio_passed else "FAILED")
print("Teacher gradient tensors  : 0")
print("Optimizer step performed  : False")
print("Progress                  : 6 / 8 code cells")
print("Next                      : mixed-precision and resource audit")


PHASE 05B CELL 6 GRADIENT-RATIO GATE
Base caption CE           : 11.300378799438477
Normalized feature loss   : 2.006793737411499
Base encoder grad norm    : 6.596478462219238
Weighted feature grad norm: 0.1684090942144394
Gradient ratio            : 0.025530151455657433
Accepted interval         : [0.05, 0.50]
Gate                      : FAILED
Teacher gradient tensors  : 0
Optimizer step performed  : False
Progress                  : 6 / 8 code cells
Next                      : mixed-precision and resource audit


## Cell 7 — FP16 safety, repeatability và resource envelope

Chạy forward autocast lặp lại, không cập nhật trọng số. Kiểm tra T4 memory envelope.


In [7]:
STARTED = time.perf_counter()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

def fp16_probe():
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
        source = teacher_source(TEACHER_IMAGES)
        aligned_tokens = ALIGNMENT(source)
        student_tokens = STUDENT_ENCODER(STUDENT_IMAGES)
        logits = STUDENT_DECODER(HUMAN_INPUT, student_tokens, HUMAN_MASK)
    loss = normalized_feature_loss(student_tokens, aligned_tokens)
    return aligned_tokens.detach().float(), student_tokens.detach().float(), logits.detach().float(), loss.detach().float()

set_deterministic(SEED)
fp_a = fp16_probe()
set_deterministic(SEED)
fp_b = fp16_probe()
repeat_errors = [float((a - b).abs().max().item()) for a, b in zip(fp_a[:3], fp_b[:3])]
loss_repeat_error = float((fp_a[3] - fp_b[3]).abs().item())
all_finite = all(bool(torch.isfinite(x).all().item()) for x in fp_a)
peak_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
resource_passed = bool(all_finite and max(repeat_errors + [loss_repeat_error]) == 0.0 and peak_mib < 14500.0)

write_json(AUDIT_DIR / "phase5b_cell07_fp16_resource.json", {
    "schema_version": "phase5b_cell07_v1", "created_at_utc": now_utc(),
    "status": "passed" if resource_passed else "failed", "mixed_precision": "FP16 autocast",
    "all_tensors_finite": all_finite, "repeat_errors": repeat_errors,
    "loss_repeat_error": loss_repeat_error, "peak_gpu_memory_mib": peak_mib,
    "t4_memory_limit_mib": 14500.0, "resource_gate_passed": resource_passed,
    "optimizer_step_performed": False, "training_started": False,
    "test_set_status": "sealed", "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 72)
print("PHASE 05B CELL 7 STATUS")
print("=" * 72)
print("FP16 tensors finite       :", all_finite)
print("Repeat errors             :", repeat_errors, "| loss", loss_repeat_error)
print("Peak GPU memory           :", round(peak_mib, 2), "MiB")
print("T4 resource gate          :", "PASSED" if resource_passed else "FAILED")
print("Optimizer step performed  : False")
print("Full training started     : False")
print("Progress                  : 7 / 8 code cells")
print("Next                      : package decision and authorize Phase 05C")


PHASE 05B CELL 7 STATUS
FP16 tensors finite       : True
Repeat errors             : [0.0, 0.0, 0.0] | loss 0.0
Peak GPU memory           : 1629.76 MiB
T4 resource gate          : PASSED
Optimizer step performed  : False
Full training started     : False
Progress                  : 7 / 8 code cells
Next                      : package decision and authorize Phase 05C


## Cell 8 — Compatibility decision, manifest và Phase 05C authorization

Chỉ tạo authorization cho 05C khi tất cả gate định trước đều đạt.
Nếu không đạt, marker ghi `blocked` và nhánh feature dừng đúng fallback Phase 05A.


In [8]:
STARTED = time.perf_counter()
COMPATIBILITY_PASSED = bool(ratio_passed and resource_passed)
decision = {
    "schema_version": "phase5b_expansionnet_s1_feature_compatibility_v1",
    "created_at_utc": now_utc(),
    "status": "compatible" if COMPATIBILITY_PASSED else "incompatible",
    "teacher": "Teacher A — ExpansionNet v2 pretrained",
    "student": "S1 — MobileNetV3-Large seed 42",
    "teacher_feature_source": "input_linear(swin_transf(image)) [B,144,512]",
    "spatial_alignment": "adaptive average pooling 12x12 -> 7x7",
    "projection": "Linear(512,256)+LayerNorm",
    "identity_projection": False,
    "loss": "token-wise squared L2 distance after L2 normalization",
    "lambda_feature": LAMBDA_FEATURE,
    "gradient_ratio": gradient_ratio,
    "gradient_ratio_gate": [GRADIENT_RATIO_MIN, GRADIENT_RATIO_MAX],
    "gradient_ratio_gate_passed": ratio_passed,
    "fp16_resource_gate_passed": resource_passed,
    "teacher_frozen": True,
    "teacher_stop_gradient": True,
    "direct_logit_kd_allowed": False,
    "optimizer_step_performed": False,
    "full_training_started": False,
    "phase05c_authorized": COMPATIBILITY_PASSED,
    "fallback_if_incompatible": study["fallback_if_feature_incompatible"],
    "test_set_status": "sealed",
    "test_set_used": False,
}
DECISION_PATH = CONFIG_DIR / "phase5b_expansionnet_s1_feature_compatibility_contract.json"
write_json(DECISION_PATH, decision)

if COMPATIBILITY_PASSED:
    auth_payload = {
        "schema_version": "phase5c_expansionnet_s1_feature_readiness_authorization_v1",
        "created_at_utc": now_utc(), "status": "authorized",
        "teacher": decision["teacher"], "student": decision["student"],
        "authorized_next": "Phase 05C training readiness only",
        "feature_contract": {
            "relative_path": DECISION_PATH.relative_to(OUTPUT_ROOT).as_posix(),
            "sha256": sha256_file(DECISION_PATH),
        },
        "lambda_feature": LAMBDA_FEATURE,
        "allowed_seed": 42,
        "full_training_authorized": False,
        "test_set_status": "sealed", "test_set_used": False,
    }
    AUTH_OUT = OUTPUT_ROOT / "PHASE5C_EXPANSIONNET_S1_FEATURE_READINESS_AUTHORIZED.json"
    write_json(AUTH_OUT, auth_payload)
else:
    AUTH_OUT = OUTPUT_ROOT / "PHASE5C_EXPANSIONNET_S1_FEATURE_READINESS_BLOCKED.json"
    write_json(AUTH_OUT, {
        "schema_version": "phase5c_expansionnet_s1_feature_readiness_blocked_v1",
        "created_at_utc": now_utc(), "status": "blocked",
        "reason": "Phase 05B prespecified compatibility gate failed",
        "full_training_authorized": False,
        "test_set_status": "sealed", "test_set_used": False,
    })

audit_files = sorted(AUDIT_DIR.glob("*.json"))
assert len(audit_files) == 7
artifact_files = audit_files + sorted(CONFIG_DIR.glob("*.json")) + sorted(RESULT_DIR.glob("*.json"))
manifest_rows = []
for path in artifact_files:
    manifest_rows.append({
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
MANIFEST_CSV = MANIFEST_DIR / "phase5b_expansionnet_s1_feature_manifest.csv"
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["relative_path", "size_bytes", "sha256"])
    writer.writeheader()
    writer.writerows(manifest_rows)
SHA_PATH = MANIFEST_DIR / "SHA256SUMS"
SHA_PATH.write_text(
    "".join(f"{row['sha256']}  {row['relative_path']}\n" for row in manifest_rows),
    encoding="utf-8",
)
marker = {
    "schema_version": "phase5b_expansionnet_s1_feature_preflight_complete_v1",
    "created_at_utc": now_utc(),
    "status": "complete" if COMPATIBILITY_PASSED else "complete_incompatible",
    "compatibility": "passed" if COMPATIBILITY_PASSED else "failed",
    "teacher": decision["teacher"], "student": decision["student"],
    "gradient_ratio": gradient_ratio, "gradient_ratio_gate_passed": ratio_passed,
    "fp16_resource_gate_passed": resource_passed,
    "phase05c_authorized": COMPATIBILITY_PASSED,
    "next_marker": {
        "relative_path": AUTH_OUT.relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(AUTH_OUT),
    },
    "manifest": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(OUTPUT_ROOT).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV), "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": SHA_PATH.relative_to(OUTPUT_ROOT).as_posix(),
            "sha256": sha256_file(SHA_PATH),
        },
    },
    "optimizer_step_performed": False, "full_training_started": False,
    "test_set_status": "sealed", "test_set_used": False,
}
MARKER_PATH = OUTPUT_ROOT / "PHASE5B_EXPANSIONNET_S1_FEATURE_PREFLIGHT_COMPLETE.json"
write_json(MARKER_PATH, marker)

print("=" * 72)
print("PHASE 05B FINAL STATUS")
print("=" * 72)
print("Status                    :", marker["status"].upper())
print("Feature compatibility     :", marker["compatibility"].upper())
print("Gradient-ratio gate       :", "PASSED" if ratio_passed else "FAILED")
print("FP16/resource gate        :", "PASSED" if resource_passed else "FAILED")
print("Phase 05C authorized      :", COMPATIBILITY_PASSED)
print("Manifest rows             :", len(manifest_rows))
print("Optimizer step performed  : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Marker                    :", MARKER_PATH)
print("Next                      :", (
    "run Phase 05C training readiness"
    if COMPATIBILITY_PASSED
    else "stop feature branch and apply Phase 05A fallback"
))
print("Progress                  : 8 / 8 code cells")


PHASE 05B FINAL STATUS
Status                    : COMPLETE_INCOMPATIBLE
Feature compatibility     : FAILED
Gradient-ratio gate       : FAILED
FP16/resource gate        : PASSED
Phase 05C authorized      : False
Manifest rows             : 9
Optimizer step performed  : False
Full training started     : False
Test Set status           : SEALED
Marker                    : /kaggle/working//phase5_teacher_a_expansionnet_s1_feature_preflight/PHASE5B_EXPANSIONNET_S1_FEATURE_PREFLIGHT_COMPLETE.json
Next                      : stop feature branch and apply Phase 05A fallback
Progress                  : 8 / 8 code cells
